In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from copy import deepcopy

def chunk_documents(documents, chunk_size=1000, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )

    chunked_docs = []

    for parent_doc_id, doc in enumerate(documents):
        chunks = splitter.split_documents([doc])

        for chunk_id, chunk in enumerate(chunks):
            #  Important: copy metadata so chunks don't share references
            chunk.metadata = deepcopy(chunk.metadata)

            # Add chunk-level metadata
            chunk.metadata.update({
                "chunk_id": chunk_id,
                "parent_doc_id": parent_doc_id,
            })

            chunked_docs.append(chunk)

    return chunked_docs

# if __name__== "__main__":
#     main()

In [2]:
import pickle as pkl

In [3]:
with open(r"C:\Users\Shaaf\Desktop\Data Science\Practice Projects\PDF_Video Reader\Cache\cached_documents.pkl", "rb") as f:
    documents = pkl.load(f)


In [4]:
chunked_docs = chunk_documents(documents, chunk_size=1000, chunk_overlap=200)

In [6]:
len(chunked_docs)

802

In [ ]:

from langchain.embeddings.base import Embeddings
from sentence_transformers import SentenceTransformer
import os
from langchain_community.vectorstores import FAISS, Chroma
import logging
from pathlib import Path

BASE_DIR = Path().resolve().parent.parent
CACHE_DIR = BASE_DIR/"Cache"
CACHE_DIR.mkdir(exist_ok=True)

logger = logging.getLogger(__name__)
class HFEmbeddings(Embeddings):
    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        return self.model.encode(texts,normalize_embeddings=True,show_progress_bar=False).tolist()

    def embed_query(self, text):
        return self.model.encode(text,normalize_embeddings=True).tolist()
    


embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
        trust_remote_code=True)


# ---------------- FAISS Handler ----------------
def build_or_load_faiss(chunked_documents, persist_dir=CACHE_DIR/"faiss_index", needs_refresh=False):
    embeddings = HFEmbeddings(embedding_model)
    persist_path = Path(persist_dir)
    index_file = persist_path / "index.faiss"
    # We rebuild if the folder doesn't exist OR if our fingerprint says data changed
    if not index_file.exists() or needs_refresh:
        try:
            persist_path.mkdir(parents=True, exist_ok=True)
            logger.info("Creating new FAISS vectorstore...")
            vectorstore = FAISS.from_documents(
                documents=chunked_documents,
                embedding=embeddings
            )
            # save_local creates the folder for you!
            vectorstore.save_local(str(persist_path))
        except Exception as e:
            logger.error(f"Failed to create FAISS vectorstore: {e}")
            raise e
    else:
        try:
            logger.info("Loading existing FAISS vectorstore...")
            vectorstore = FAISS.load_local(
                str(persist_path), 
                embeddings,
                allow_dangerous_deserialization=True
            )
            # IMPORTANT: We removed .add_documents() here to prevent duplicates
        except Exception as e:
            logger.error(f"Failed to load FAISS vectorstore: {e}")
            raise e

    return vectorstore




In [17]:
# Demo usage
from src.document_reader import load_all_documents
from src.Chunking import chunk_documents

files_path = BASE_DIR / "PDF_Samples"
documents = load_all_documents(files_path)
chunked_docs = chunk_documents(documents)
faiss_store = build_or_load_faiss(chunked_docs)
#     logger.info(f"FAISS vectorstore contains {faiss_store.index.ntotal} vectors.")

2026-10-25 14:22:54,006 - INFO - Folder contents unchanged. Loading documents from cache...
2026-10-25 14:22:54,057 - INFO - Loading existing FAISS vectorstore...


In [ ]:
from sentence_transformers import CrossEncoder
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

##function for similarity reranking


reranker = CrossEncoder("BAAI/bge-reranker-base")
import logging
logger = logging.getLogger(__name__)
logger.info("Reranker model loaded successfully.")
logger.info("Defining similarity_reranker function.")
def similarity_reranker(chunked_docs,vectorstore,query,search_type):
    try:
        bm25_retriever = BM25Retriever.from_documents(chunked_docs)
        bm25_retriever.k = 10

        if vectorstore is None:
            raise ValueError("Vectorstore is not initialized.")
                # Create BM25 before the branches so Hybrid can use it
        
        elif search_type=="Semantic Search":
            logger.info("Starting semantic search and reranking.")
            reranked_results = vectorstore.similarity_search(query=query, k=10)
            pairs = [(query, doc.page_content) for doc in reranked_results]
            scores = reranker.predict(pairs)
            reranked_docs = sorted(
                zip(reranked_results, scores),
                key=lambda x: x[1],
                reverse=True)

        elif search_type=="Keyword Search":
            logger.info("Starting keyword search and reranking.")
            reranked_results = bm25_retriever.invoke(query)
            pairs = [(query, doc.page_content) for doc in reranked_results]
            scores = reranker.predict(pairs)
            reranked_docs = sorted(
                zip(reranked_results, scores),
                key=lambda x: x[1],
                reverse=True)

        elif search_type=="Hybrid Search":
            logger.info("Starting hybrid search and reranking.")
            ensemble_retriever = EnsembleRetriever(retrievers=[vectorstore.as_retriever(search_kwargs={"k": 10}), bm25_retriever],weights=[0.5, 0.5])
            reranked_results = ensemble_retriever.invoke(query)
            pairs = [(query, doc.page_content) for doc in reranked_results]
            scores = reranker.predict(pairs)
            reranked_docs = sorted(
                zip(reranked_results, scores),
                key=lambda x: x[1],
                reverse=True)

        return reranked_docs
    
    except Exception as e:
        logger.error(f"Error during similarity search and reranking: {e}")
        raise e



2026-10-25 15:49:10,234 - INFO - Use pytorch device: cpu
2026-10-25 15:49:10,535 - INFO - Reranker model loaded successfully.
2026-10-25 15:49:10,537 - INFO - Defining similarity_reranker function.


In [23]:
reranked_docs=faiss_store.similarity_search(query="Explain the time series analysis",k=10)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [26]:
query="Explain the time series analysis"
pairs = [(query, doc.page_content) for doc in reranked_docs]

In [27]:
reranker.predict(pairs)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

array([0.92907417, 0.6611779 , 0.68761617, 0.13872589, 0.91481733,
       0.48011485, 0.8983423 , 0.34906557, 0.24234757, 0.94163287],
      dtype=float32)

In [32]:
similarity_reranker(faiss_store,"Explain the time series analysis","Hybrid Search")

2026-10-25 15:50:28,146 - INFO - Starting hybrid search and reranking.
2026-10-25 15:50:28,147 - ERROR - Error during similarity search and reranking: cannot access local variable 'bm25_retriever' where it is not associated with a value


UnboundLocalError: cannot access local variable 'bm25_retriever' where it is not associated with a value

In [40]:
from src.document_reader import load_all_documents
from src.Chunking import chunk_documents
from src.Embedding import build_or_load_faiss
from src.similarity_reranker import similarity_reranker
from src.Agentic_RAG import agentic_rag
import os
from pathlib import Path
BASE_DIR = Path().resolve()
files_path = BASE_DIR / "PDF_Samples"

def main():
    # Load and prepare documents
    documents = load_all_documents(files_path)
    chunked_docs = chunk_documents(documents)
    embed_store = build_or_load_faiss(chunked_docs)
    reranked_docs = similarity_reranker(chunked_docs, embed_store, "Explain the time series analysis", "Hybrid Search")
    print("Reranked Documents:", reranked_docs)




In [41]:
BASE_DIR

WindowsPath('C:/Users/Shaaf/Desktop/Data Science/Practice Projects/PDF_Video Reader')

In [42]:
main()

2026-10-25 16:51:38,452 - INFO - Folder contents unchanged. Loading documents from cache...
2026-10-25 16:51:38,500 - INFO - Loading existing FAISS vectorstore...
2026-10-25 16:51:38,569 - INFO - Starting hybrid search and reranking.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Reranked Documents: [(Document(id='b96de5db-920c-4d2f-925d-0e81b8bdefe0', metadata={'producer': 'Adobe Acrobat Pro 11.0.20 Paper Capture Plug-in', 'creator': 'PScript5.dll Version 5.2', 'creationdate': '2005-08-18T01:21:09+03:00', 'author': 'dbsubasi', 'moddate': '2017-12-23T12:10:40+05:00', 'title': 'DjVu Document', 'source': 'Analysis_of_Time_Series_An_Introduction.pdf', 'total_pages': 293, 'page': 20, 'page_label': '21', 'path': 'C:\\Users\\Shaaf\\Desktop\\Data Science\\Practice Projects\\PDF_Video Reader\\PDF_Samples\\Analysis_of_Time_Series_An_Introduction.pdf', 'file_type': 'pdf', 'doc_id': '71e3cc67c62e660eaa62b199f044c97ba1650bf8', 'chunk_id': 0, 'parent_doc_id': 21}, page_content="Transformations 11 \n2.3 THE TIME PLOT \nAfter getting background information and carefully defining objectives, the \nfirst, and most important, step in any time-series analysis is to plot the \nobservations against time. This graph should show up important features of \nthe series such as trend, se

In [44]:
for id, doc in enumerate(reranked_docs):
    print(doc)

page_content='deterministic. But most time series are stochastic in that the future is only 
partly determined by past values, so that exact predictions are impossible and 
must be replaced by the idea that future values have a probability distribution 
which is conditioned by a knowledge of past values. 
1.3 OBJECTIVES OF TIME-SERIES ANALYSIS 
There are several possible objectives in analysing a time series. These 
objectives may be classified as description, explanation, prediction and 
control, and will be considered in turn. 
(a) Description 
When presented with a time series, the first step in the analysis is usually to plot 
the data and to obtain simple descriptive measures of the main properties of 
the series as described in Chapter 2. For example, looking at Figure 1.3 it can 
be seen that there is a regular seasonal effect, with sales 'high' in winter and 
'low' in summer It also looks as though annual sales are increasing (i e show' metadata={'producer': 'Adobe Acrobat Pro 